# MNIST 手写数字识别 — 卷积神经网络（CNN）多分类

> **项目目标：** 使用 TensorFlow/Keras 构建 CNN 模型，对 MNIST 手写数字 0-9 进行分类识别。
>
> **本 Notebook 章节：**
> 1. 环境准备与数据加载
> 2. 数据预处理
> 3. 数据可视化
> 4. 模型搭建（逐层详解）
> 5. 模型编译
> 6. 模型训练
> 7. 模型评估（混淆矩阵）
> 8. 预测验证
> 9. 保存与加载模型
> 10. 调参实验区

---

## 1. 环境准备与数据加载

首先导入所需的库，然后从 `mnist/mnist.pkl.gz` 加载数据集。
该数据集包含 70000 张 28×28 的灰度手写数字图片，分为 0-9 共 10 个类别。

In [ ]:
# ========== 1. 环境准备与数据加载 ==========
import numpy as np
import matplotlib.pyplot as plt
import pickle
import gzip
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 设置随机种子，保证结果可复现
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow 版本: {tf.__version__}")
print(f"NumPy 版本: {np.__version__}")

# 设置中文字体支持（Windows）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 加载 MNIST 数据集
data_path = os.path.join('..', 'mnist', 'mnist.pkl.gz')
with gzip.open(data_path, 'rb') as f:
    # Python 3 兼容性处理
    training_data, validation_data, test_data = pickle.load(f, encoding='latin1')

X_train_raw, y_train_raw = training_data    # 训练集
X_val_raw, y_val_raw = validation_data       # 验证集
X_test_raw, y_test_raw = test_data           # 测试集

print(f"\n训练集样本数: {len(X_train_raw):,}")
print(f"验证集样本数: {len(X_val_raw):,}")
print(f"测试集样本数: {len(X_test_raw):,}")
print(f"图像尺寸: {X_train_raw.shape[1]}×{X_train_raw.shape[2] if len(X_train_raw.shape) > 1 else '扁平'}")
print(f"单个样本的原始形状: {X_train_raw[0].shape}")

## 2. 数据预处理

对原始数据做以下处理：
1. **形状变换：** 将扁平向量 (784,) 还原为 28×28 的图像矩阵，并增加通道维度 → (28, 28, 1)
2. **归一化：** 将像素值从 [0, 255] 缩放到 [0, 1]
3. **标签处理：** Keras 的 `sparse_categorical_crossentropy` 接受整数标签，无需 one-hot 编码
4. **合并训练集和验证集：** 将原始训练集和验证集合并，然后按 9:1 重新划分

In [ ]:
# ========== 2. 数据预处理 ==========

# 2.1 形状变换：扁平向量 (784,) → 28×28 灰度图
X_train_raw = X_train_raw.reshape(-1, 28, 28, 1)
X_val_raw   = X_val_raw.reshape(-1, 28, 28, 1)
X_test_raw  = X_test_raw.reshape(-1, 28, 28, 1)

# 2.2 归一化：像素值 0-255 → 0-1
X_train = X_train_raw.astype('float32') / 255.0
X_val   = X_val_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32') / 255.0

y_train = np.array(y_train_raw)
y_val   = np.array(y_val_raw)
y_test  = np.array(y_test_raw)

# 2.3 合并训练集和验证集，重新划分
X_all = np.concatenate([X_train, X_val], axis=0)
y_all = np.concatenate([y_train, y_val], axis=0)

from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

print(f"处理后训练集: {X_train.shape}, 标签: {y_train.shape}")
print(f"处理后验证集: {X_valid.shape}, 标签: {y_valid.shape}")
print(f"处理后测试集: {X_test.shape}, 标签: {y_test.shape}")
print(f"\n像素值范围: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"标签类别: {np.unique(y_train)}")
print(f"每个类别样本数: {np.bincount(y_train.astype(int))}")

## 3. 数据可视化

随机抽取 25 张训练样本，以 5×5 网格展示，每个子图显示图像及其对应标签。这样可以直观了解数据的样子。

In [ ]:
# ========== 3. 数据可视化 ==========
fig, axes = plt.subplots(5, 5, figsize=(8, 8))
fig.suptitle('MNIST 训练集样本展示（随机抽取 25 张）', fontsize=16, fontweight='bold')

# 随机选取 25 个不同索引
indices = np.random.choice(len(X_train), 25, replace=False)

for i, ax in enumerate(axes.flat):
    idx = indices[i]
    ax.imshow(X_train[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'标签: {y_train[idx]}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. 模型搭建

下面构建一个经典的 CNN（卷积神经网络）用于图像分类。逐层说明：

| 层 | 类型 | 参数 | 说明 |
|----|------|------|------|
| 1 | Conv2D | 32 个 3×3 卷积核 | 提取低级特征（边缘、纹理） |
| 2 | MaxPooling2D | 2×2 窗口 | 下采样，减少参数量 |
| 3 | Conv2D | 64 个 3×3 卷积核 | 提取高级特征（组合形状） |
| 4 | MaxPooling2D | 2×2 窗口 | 进一步下采样 |
| 5 | Flatten | - | 展平为 1D 向量 |
| 6 | Dense | 128 神经元 + ReLU | 全连接层，综合特征 |
| 7 | Dropout | 0.5 | 随机丢弃 50% 神经元，防止过拟合 |
| 8 | Dense | 10 神经元 + Softmax | 输出层，10 类概率分布 |

> **思考：为什么用 Conv2D 而不是全连接？** 卷积层能保留图像的空间结构信息，参数量更少。如果直接用全连接层处理 28×28=784 维输入，参数量会非常大且丢失空间关系。

In [ ]:
# ========== 4. 模型搭建 ==========
model = keras.Sequential([
    # 第1层：卷积层 — 32个3×3卷积核，ReLU激活，输入28×28灰度图
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), name='conv1'),
    # 第2层：池化层 — 2×2最大池化，将特征图尺寸减半
    layers.MaxPooling2D((2, 2), name='pool1'),
    
    # 第3层：卷积层 — 64个3×3卷积核，提取更高级特征
    layers.Conv2D(64, (3, 3), activation='relu', name='conv2'),
    # 第4层：池化层 — 再次减半特征图
    layers.MaxPooling2D((2, 2), name='pool2'),
    
    # 第5层：展平 — 将 5×5×64=1600 的多维特征图拉平为1D向量
    layers.Flatten(name='flatten'),
    
    # 第6层：全连接层 — 128个神经元，综合所有特征
    layers.Dense(128, activation='relu', name='fc1'),
    # 第7层：Dropout — 训练时随机丢弃50%神经元，防止过拟合
    layers.Dropout(0.5, name='dropout'),
    
    # 第8层：输出层 — 10个神经元对应0-9数字，Softmax输出概率分布
    layers.Dense(10, activation='softmax', name='output')
], name='MNIST_CNN')

# 打印模型结构概要
model.summary()

### 4.1 参数量分析

我们来手动验算一下各层的参数量：

- **Conv2D(32, 3×3)：** 每个卷积核 3×3 = 9 个权重 + 1 个偏置 = 10 个参数/核。32 个核 × 10 = **320** 参数
- **MaxPooling2D：** 无参数
- **Conv2D(64, 3×3)：** 输入有 32 个通道，每个核要跨所有输入通道：32 × 3 × 3 + 1 = 289。64 个核 × 289 = **18,496** 参数
- **Flatten：** 无参数
- **Dense(128)：** 1600 × 128 + 128 = **204,928** 参数（最大的一层！）
- **Dropout：** 无参数
- **Dense(10)：** 128 × 10 + 10 = **1,290** 参数

总计约 22.5 万参数，对于 MNIST 任务而言是一个轻量模型。

## 5. 模型编译

编译时需要指定三个要素：
1. **优化器（Optimizer）：** Adam — 自适应学习率的梯度下降变体，收敛快且稳定
2. **损失函数（Loss）：** Sparse Categorical Crossentropy — 适用于整数标签的多分类问题
3. **评估指标（Metrics）：** Accuracy — 分类准确率

> **思考：为什么用 Sparse Categorical Crossentropy 而不是 Categorical Crossentropy？** 因为我们的标签是整数（0-9），前者不需要手动做 one-hot 编码。

In [ ]:
# ========== 5. 模型编译 ==========
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("模型编译完成！")
print(f"优化器: Adam (lr=0.001)")
print(f"损失函数: Sparse Categorical Crossentropy")
print(f"评估指标: Accuracy")

## 6. 模型训练

现在开始训练模型。训练过程中会实时显示每个 epoch 的 loss 和 accuracy。

- **epochs = 10：** 遍历全量数据 10 次
- **batch_size = 64：** 每次取 64 张图片计算梯度并更新权重
- **validation_data：** 每个 epoch 结束后在验证集上评估，用于观察过拟合

In [ ]:
# ========== 6. 模型训练 ==========
EPOCHS = 10
BATCH_SIZE = 64

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid, y_valid),
    verbose=1
)

print(f"\n训练完成！最终训练准确率: {history.history['accuracy'][-1]:.4f}")
print(f"最终验证准确率: {history.history['val_accuracy'][-1]:.4f}")

### 6.1 训练曲线可视化

绘制两条曲线：
1. **Loss 曲线（左）：** 展示训练损失和验证损失随 epoch 的变化。如果验证损失上升而训练损失下降，说明出现过拟合。
2. **Accuracy 曲线（右）：** 展示训练准确率和验证准确率的变化趋势。

In [ ]:
# ========== 6.1 训练曲线 ==========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
ax1.plot(history.history['loss'], 'o-', label='训练损失 (Training Loss)', linewidth=2, markersize=4)
ax1.plot(history.history['val_loss'], 's-', label='验证损失 (Validation Loss)', linewidth=2, markersize=4)
ax1.set_title('模型损失曲线 (Loss)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy 曲线
ax2.plot(history.history['accuracy'], 'o-', label='训练准确率 (Training Acc)', linewidth=2, markersize=4)
ax2.plot(history.history['val_accuracy'], 's-', label='验证准确率 (Validation Acc)', linewidth=2, markersize=4)
ax2.set_title('模型准确率曲线 (Accuracy)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 模型评估

在**测试集**上评估模型的最终表现。测试集数据模型从未见过，能真实反映泛化能力。

还会绘制**混淆矩阵**（Confusion Matrix），直观展示每个数字的识别正确率和常见误判模式。

In [ ]:
# ========== 7. 模型评估 ==========
# 7.1 测试集评估
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"="*50)
print(f"测试集 Loss:     {test_loss:.4f}")
print(f"测试集 Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"="*50)

# 7.2 获取所有预测结果
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
# 7.3 混淆矩阵
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.title('混淆矩阵 (Confusion Matrix)', fontsize=16, fontweight='bold')
plt.xlabel('预测标签 (Predicted)')
plt.ylabel('真实标签 (True)')
plt.tight_layout()
plt.show()

# 打印每个类别的准确率
print("\n各类别识别准确率:")
for i in range(10):
    tp = cm[i, i]
    total = cm[i, :].sum()
    print(f"  数字 {i}: {tp}/{total} = {tp/total*100:.2f}%")

### 7.1 分类报告

展示精确率（Precision）、召回率（Recall）和 F1-Score。

In [ ]:
# 7.4 分类报告
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)])
print("分类报告 (Classification Report):")
print(report)

## 8. 预测验证

随机抽取 10 张测试集图片，分别显示：真实标签、模型预测结果、以及是否正确。预测正确的用绿色标注，错误的用红色标注。这样可以直观感受模型的实际表现。

In [ ]:
# ========== 8. 预测验证 ==========
import random

# 随机抽取 12 张图片
sample_indices = random.sample(range(len(X_test)), 12)

fig, axes = plt.subplots(3, 4, figsize=(12, 8))
fig.suptitle('模型预测结果验证（绿色=正确，红色=错误）', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    idx = sample_indices[i]
    img = X_test[idx].reshape(28, 28)
    true_label = y_test[idx]
    pred_label = y_pred[idx]
    is_correct = (true_label == pred_label)
    
    ax.imshow(img, cmap='gray')
    color = 'green' if is_correct else 'red'
    ax.set_title(f'真实: {true_label} | 预测: {pred_label}', color=color, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

### 8.1 错误案例分析

找出模型预测错误的样本，展示几个最具代表性的错误案例。分析模型在哪些数字之间容易混淆（例如 4 和 9，3 和 8 等）。

In [ ]:
# ========== 8.1 错误案例分析 ==========
error_indices = np.where(y_pred != y_test)[0]
print(f"测试集中共有 {len(error_indices)} 个错误预测 ({len(error_indices)/len(y_test)*100:.2f}%)")

# 展示前 9 个错误案例
n_show = min(9, len(error_indices))
if n_show > 0:
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    fig.suptitle(f'模型错误预测案例（共 {len(error_indices)} 个错误）', fontsize=16, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        if i < n_show:
            idx = error_indices[i]
            ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
            ax.set_title(f'真实: {y_test[idx]} → 预测: {y_pred[idx]}', color='red', fontsize=11)
        else:
            ax.axis('off')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("完美！模型没有预测错误。")

## 9. 保存与加载模型

将训练好的模型保存为 `.h5` 文件，供后续 GUI 应用加载使用。
保存后会重新加载模型并验证准确率是否一致。

In [ ]:
# ========== 9. 保存与加载模型 ==========
MODEL_PATH = os.path.join('.', 'mnist_cnn.h5')

# 9.1 保存模型
model.save(MODEL_PATH)
print(f"模型已保存至: {os.path.abspath(MODEL_PATH)}")
print(f"文件大小: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

In [ ]:
# 9.2 加载模型并验证
loaded_model = keras.models.load_model(MODEL_PATH)
loaded_loss, loaded_acc = loaded_model.evaluate(X_test, y_test, verbose=0)

print(f"原始模型测试准确率: {test_acc:.4f}")
print(f"加载模型测试准确率: {loaded_acc:.4f}")
print(f"准确率是否一致: {'✓ 一致' if abs(test_acc - loaded_acc) < 0.0001 else '✗ 不一致，请检查！'}")

# 验证单张图片推理
sample_img = X_test[0:1]  # 取第一张，保持 batch 维度
pred = loaded_model.predict(sample_img, verbose=0)
pred_digit = np.argmax(pred)
pred_conf = np.max(pred)
print(f"\n单张推理验证: 真实标签={y_test[0]}, 预测结果={pred_digit}, 置信度={pred_conf:.4f}")

## 10. 调参实验区

下面是一些你可以尝试的调参方向，每个方向都可以复制上面的代码并修改参数后重新运行，观察对模型性能的影响。

### 可调整的超参数：

| 参数 | 当前值 | 建议尝试 |
|------|--------|----------|
| 学习率 (learning_rate) | 0.001 | 0.0001, 0.01 |
| 卷积核数量 | (32, 64) | (16, 32), (64, 128) |
| 卷积核大小 | 3×3 | 5×5 |
| Dropout 比例 | 0.5 | 0.3, 0.7 |
| 全连接层神经元数 | 128 | 64, 256 |
| Batch Size | 64 | 32, 128 |
| Epochs | 10 | 15, 20 |
| 是否增加 BN 层 | 无 | BatchNormalization 加在 Conv 后 |
| 激活函数 | ReLU | LeakyReLU, ELU |

### 实验模板

在下方复制上面的「模型搭建 + 编译 + 训练」代码，修改你想调整的参数，重新运行即可。
建议每次只改一个参数，保持其他参数不变，这样能清楚看到该参数的影响。

In [ ]:
# ========== 10. 调参实验区 ==========
# 💡 提示：复制上面的模型代码到这里，修改参数后运行

# 示例：尝试增加卷积核数量 (32,64) → (64,128)

# model_v2 = keras.Sequential([
#     layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),  # 改为 64
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(128, (3, 3), activation='relu'),                            # 改为 128
#     layers.MaxPooling2D((2, 2)),
#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.5),
#     layers.Dense(10, activation='softmax')
# ])
#
# model_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_v2 = model_v2.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_valid, y_valid))
# test_loss_v2, test_acc_v2 = model_v2.evaluate(X_test, y_test, verbose=0)
# print(f"版本2 测试准确率: {test_acc_v2:.4f}")

print("调参实验区已就绪，请取消上方注释并修改参数后运行。")
print("记得对比两次实验的训练曲线和最终准确率。")

---

## 总结

恭喜！你已经完成了一个完整的 CNN 手写数字识别项目：

1. ✅ 加载并预处理了 MNIST 数据集
2. ✅ 可视化查看了数据样本
3. ✅ 搭建了一个 2 层卷积 + 2 层全连接的 CNN 模型
4. ✅ 理解并分析了各层的参数量
5. ✅ 训练了模型并观察了 loss/accuracy 变化曲线
6. ✅ 在测试集上评估了模型，绘制了混淆矩阵
7. ✅ 分析了错误案例
8. ✅ 保存了模型供后续使用
9. ✅ 预留了调参实验空间

下一步：运行 `gui_app.py` 启动手写识别上位机，体验实时手写数字识别！